In [1]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import PoissonRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score

#Con este modelo buscamos predecir el resultado de un partido de fútbol entre
#dos selecciones. Para hacerlo, usamos resultados históricos de partidos
#anteriores y transforma esa información en variables numéricas que luego
#alimentan modelos estadísticos y de machine learning.

#La base principal es results.csv, que contiene partidos históricos
#con columnas como fecha, equipo local, equipo visitante, goles, torneo,
#ciudad, país y si el partido fue neutral. Nuestro archivo actual,
#el modelo carga esos datos, elimina filas incompletas y conserva
#partidos desde 2018 en adelante. Esto evita que partidos demasiado
#antiguos influyan demasiado en la predicción actual.

In [2]:
#modelo 1.2

# ==================================================
# RESUMEN EQUIPO
# ==================================================

def resumen_equipo(partidos):

    if len(partidos) == 0:

        return {

            "gf":0,
            "gc":0,

            "wins":0,
            "draws":0,
            "losses":0,

            "gd":0,

            "streak":0
        }

    gf = sum(x["gf"] for x in partidos)

    gc = sum(x["gc"] for x in partidos)

    wins = sum(
        x["resultado"] == "W"
        for x in partidos
    )

    draws = sum(
        x["resultado"] == "D"
        for x in partidos
    )

    losses = sum(
        x["resultado"] == "L"
        for x in partidos
    )

    streak = 0

    for p in reversed(partidos):

        if p["resultado"] == "W":

            streak += 1

        else:

            break

    return {

        "gf":gf,
        "gc":gc,

        "wins":wins,
        "draws":draws,
        "losses":losses,

        "gd":gf-gc,

        "streak":streak
    }

In [3]:
# ==================================================
# CONFIGURACIÓN
# ==================================================

ANIO_INICIO = 2018

TEST_SIZE = 0.20

RANDOM_STATE = 42

N_MONTECARLO = 100000

In [ ]:
# ==================================================
# CARGA  (INGESTA LIMPIA)
# ==================================================

# Leemos SIEMPRE el mismo archivo oficial. Ya no hay que renombrar
# "results (0).csv", "results (1).csv", etc. cada dia.
# Cada vez que cae un resultado nuevo, se escribe en este CSV y se
# vuelve a ejecutar el notebook.
df_raw = pd.read_csv("results_actualizado_mundial_2026.csv")

df_raw["date"] = pd.to_datetime(df_raw["date"])

# --------------------------------------------------
# Separar PARTIDOS JUGADOS vs PARTIDOS POR JUGAR
# --------------------------------------------------
# El CSV ya trae los fixtures del Mundial 2026 con el marcador vacio.
# - jugados   -> sirven para entrenar y para calcular ELO / forma.
# - por_jugar -> es el calendario que queremos predecir.

tiene_marcador = (
    df_raw["home_score"].notna()
    & df_raw["away_score"].notna()
)

datos_completos = (
    df_raw[["date", "home_team", "away_team"]]
    .notna()
    .all(axis=1)
)

df_full = (
    df_raw[tiene_marcador & datos_completos]
    .sort_values("date")
    .reset_index(drop=True)
    .copy()
)

por_jugar = (
    df_raw[~tiene_marcador & datos_completos]
    .sort_values("date")
    .reset_index(drop=True)
    .copy()
)

# IMPORTANTE (mejora 2 - ELO):
# df_full contiene TODA la historia (desde 1872). El ELO se va a calcular
# sobre toda esta historia para que los ratings lleguen "asentados" a 2018,
# en vez de reiniciar a todos en 1500 (lo que distorsionaba el ranking).
# El recorte a partir de ANIO_INICIO se hace DESPUES, al final del ELO.

print("Partidos jugados (total historia):", len(df_full))
print("Fecha minima:", df_full["date"].min())
print("Fecha maxima:", df_full["date"].max())
print("Partidos por jugar (a predecir):", len(por_jugar))

# A partir de aqui el resto del pipeline usa 'df'.
df = df_full


In [5]:
# ==================================================
# LOCALÍA
# ==================================================

#Neutral = No: el partido se juega en casa del equipo local.
#Ejemplo: México vs Portugal en México.

#Neutral = Sí: el partido se juega en una sede neutral, donde ninguno
#tiene ventaja de localía.
#Ejemplo: México vs Portugal en un Mundial en Estados Unidos,
#Catar, Alemania, etc.

df["neutral"] = (
    df["neutral"]
    .astype(str)
    .str.lower()
)

df["neutral_match"] = (

    df["neutral"]

    .isin(
        [
            "true",
            "1"
        ]
    )

).astype(int)

df["home_advantage"] = (
    1 -
    df["neutral_match"]
)

print(
    "Partidos neutrales:",
    df["neutral_match"].sum()
)

Partidos neutrales: 2565


In [6]:
# ==================================================
# PESO TORNEOS
# ==================================================

#No todos los partidos valen igual.
#Un amistoso no debería tener el mismo impacto que un Mundial.
#Por eso el modelo asigna pesos:

#Una victoria en Mundial pesa más que una victoria en amistoso.
#Esto ayuda a que el modelo no sobrevalore selecciones que ganan
#muchos amistosos o partidos de bajo nivel.

peso_torneo = {

    "FIFA World Cup":2.0,

    "UEFA Euro":1.8,
    "Copa América":1.7,

    "FIFA World Cup qualification":1.5,

    "UEFA Nations League":1.3,
    "CONCACAF Nations League":1.2,

    "African Cup of Nations":1.2,
    "AFC Asian Cup":1.2,

    "Gold Cup":1.3,

    "Friendly":0.6

}

df["peso_torneo"] = (

    df["tournament"]

    .map(peso_torneo)

    .fillna(1.0)

)

In [ ]:
# ==================================================
# ELO DINAMICO V3.1  (historia completa + autocorreccion)
# ==================================================

# MEJORAS:
#   1) Recorre TODA la historia (1872+) -> ratings asentados a 2018.
#   2) AUTOCORRECCION DE MARGEN (estilo FiveThirtyEight): golear a un rival
#      muy inferior ya NO infla el rating. Esto corrige el sesgo que ponia
#      a selecciones con muchas goleadas regionales por encima de Brasil/
#      Alemania.
#   3) Ventana de recencia de 8 años (antes 3): da mas peso a la historia/
#      reputacion, asi Brasil/Alemania quedan en posiciones realistas, sin
#      dejar de reflejar la forma reciente.
# El recorte a ANIO_INICIO se hace AL FINAL, ya con el ELO calculado.

TAU_DAYS = 2920  # 8 años. Subir -> mas historia; bajar -> mas forma actual.

fecha_maxima = df["date"].max()

equipos = pd.unique(
    pd.concat([
        df["home_team"],
        df["away_team"]
    ])
)

elo = {equipo: 1500 for equipo in equipos}

elo_home = []
elo_away = []

for _, row in df.iterrows():

    home = row["home_team"]
    away = row["away_team"]

    elo_h = elo[home]
    elo_a = elo[away]

    elo_home.append(elo_h)
    elo_away.append(elo_a)

    # Probabilidad esperada
    exp_home = 1 / (1 + 10 ** ((elo_a - elo_h) / 400))
    exp_away = 1 - exp_home

    # Resultado real + ventaja ELO del ganador sobre el perdedor (dr)
    if row["home_score"] > row["away_score"]:
        score_home, score_away = 1, 0
        dr = elo_h - elo_a
    elif row["home_score"] < row["away_score"]:
        score_home, score_away = 0, 1
        dr = elo_a - elo_h
    else:
        score_home, score_away = 0.5, 0.5
        dr = 0.0

    # Diferencia de goles
    goal_diff = abs(row["home_score"] - row["away_score"])

    # Multiplicador de margen CON autocorreccion: si el ganador ya era muy
    # favorito (dr grande), el factor baja -> no sube por golear a un debil.
    mov_multiplier = np.log(goal_diff + 1) * (2.2 / (0.001 * dr + 2.2))

    # Recencia: 8 años
    dias = (fecha_maxima - row["date"]).days
    peso_recencia = np.exp(-dias / TAU_DAYS)

    # Peso de torneo
    torneo = row["tournament"]

    if torneo == "Friendly":
        peso_competencia = 0.60
    elif "qualification" in str(torneo).lower():
        peso_competencia = 0.90
    elif torneo in [
        "FIFA World Cup",
        "UEFA Euro",
        "Copa América",
        "African Cup of Nations",
        "AFC Asian Cup",
        "Gold Cup"
    ]:
        peso_competencia = 1.10
    else:
        peso_competencia = 0.80

    k_base = 24

    k = (
        k_base
        * peso_recencia
        * peso_competencia
        * mov_multiplier
    )

    # Limitar cambios extremos por partido
    cambio_home = np.clip(k * (score_home - exp_home), -32, 32)
    cambio_away = np.clip(k * (score_away - exp_away), -32, 32)

    elo[home] += cambio_home
    elo[away] += cambio_away

df["elo_home"] = elo_home
df["elo_away"] = elo_away
df["elo_diff"] = df["elo_home"] - df["elo_away"]

print("ELO V3.1 calculado sobre", len(df), "partidos (historia completa)")
print("Equipos:", len(elo))

# --------------------------------------------------
# RECORTE A ANIO_INICIO (ya con el ELO asentado)
# --------------------------------------------------
df = (
    df[df["date"] >= f"{ANIO_INICIO}-01-01"]
    .reset_index(drop=True)
    .copy()
)

print("Partidos usados para el modelo (>= %d): %d" % (ANIO_INICIO, len(df)))

ranking_elo = (
    pd.DataFrame({
        "PAIS": list(elo.keys()),
        "ELO": list(elo.values())
    })
    .sort_values("ELO", ascending=False)
)

ranking_elo.head(25)


In [8]:
# ==================================================
# RESUMEN EQUIPO
# ==================================================

def resumen_equipo(partidos):

    if len(partidos) == 0:

        return {

            "gf":0,
            "gc":0,

            "wins":0,
            "draws":0,
            "losses":0,

            "gd":0,

            "streak":0
        }

    gf = sum(
        x["gf"]
        for x in partidos
    )

    gc = sum(
        x["gc"]
        for x in partidos
    )

    wins = sum(
        x["resultado"] == "W"
        for x in partidos
    )

    draws = sum(
        x["resultado"] == "D"
        for x in partidos
    )

    losses = sum(
        x["resultado"] == "L"
        for x in partidos
    )

    streak = 0

    for p in reversed(partidos):

        if p["resultado"] == "W":

            streak += 1

        else:

            break

    return {

        "gf":gf,
        "gc":gc,

        "wins":wins,
        "draws":draws,
        "losses":losses,

        "gd":gf-gc,

        "streak":streak

    }

In [9]:
# ==================================================
# HEAD TO HEAD
# ==================================================

def obtener_h2h(
    historial_h2h,
    home,
    away
):

    h2h_home = 0
    h2h_away = 0
    h2h_draw = 0

    for x in historial_h2h[-5:]:

        if x["winner"] == home:

            h2h_home += 1

        elif x["winner"] == away:

            h2h_away += 1

        else:

            h2h_draw += 1

    return (
        h2h_home,
        h2h_away,
        h2h_draw
    )

In [10]:
# ==================================================
# DATASET ENTRENAMIENTO
# ==================================================

historial = {}

h2h = {}

registros = []

for _, row in df.iterrows():

    home = row["home_team"]
    away = row["away_team"]

    if home not in historial:
        historial[home] = []

    if away not in historial:
        historial[away] = []

    # ==========================
    # FORMA
    # ==========================

    home5 = resumen_equipo(
        historial[home][-5:]
    )

    home10 = resumen_equipo(
        historial[home][-10:]
    )

    away5 = resumen_equipo(
        historial[away][-5:]
    )

    away10 = resumen_equipo(
        historial[away][-10:]
    )

    # ==========================
    # H2H
    # ==========================

    clave = tuple(
        sorted(
            [home, away]
        )
    )

    historial_h2h = h2h.get(
        clave,
        []
    )

    (
        h2h_home,
        h2h_away,
        h2h_draw

    ) = obtener_h2h(

        historial_h2h,

        home,

        away

    )

    # ==========================
    # TARGET
    # ==========================

    if row["home_score"] > row["away_score"]:

        resultado = 2

    elif row["home_score"] < row["away_score"]:

        resultado = 0

    else:

        resultado = 1

    registros.append({

        # ELO

        "elo_home":
        row["elo_home"],

        "elo_away":
        row["elo_away"],

        "elo_diff":
        row["elo_diff"],

        # LOCALIA

        "home_advantage":
        row["home_advantage"],

        "neutral_match":
        row["neutral_match"],

        # TORNEO

        "peso_torneo":
        row["peso_torneo"],

        # HOME 5

        "gf5_home":
        home5["gf"],

        "gc5_home":
        home5["gc"],

        "wins5_home":
        home5["wins"],

        "draws5_home":
        home5["draws"],

        "losses5_home":
        home5["losses"],

        "gd5_home":
        home5["gd"],

        "streak_home":
        home5["streak"],

        # HOME 10

        "gf10_home":
        home10["gf"],

        "gc10_home":
        home10["gc"],

        "wins10_home":
        home10["wins"],

        "draws10_home":
        home10["draws"],

        "losses10_home":
        home10["losses"],

        "gd10_home":
        home10["gd"],

        # AWAY 5

        "gf5_away":
        away5["gf"],

        "gc5_away":
        away5["gc"],

        "wins5_away":
        away5["wins"],

        "draws5_away":
        away5["draws"],

        "losses5_away":
        away5["losses"],

        "gd5_away":
        away5["gd"],

        "streak_away":
        away5["streak"],

        # AWAY 10

        "gf10_away":
        away10["gf"],

        "gc10_away":
        away10["gc"],

        "wins10_away":
        away10["wins"],

        "draws10_away":
        away10["draws"],

        "losses10_away":
        away10["losses"],

        "gd10_away":
        away10["gd"],

        # H2H

        "h2h_home":
        h2h_home,

        "h2h_away":
        h2h_away,

        "h2h_draw":
        h2h_draw,

        # TARGETS

        "home_score":
        row["home_score"],

        "away_score":
        row["away_score"],

        "resultado":
        resultado

    })

    # ==========================
    # ACTUALIZAR HISTORIAL
    # ==========================

    if row["home_score"] > row["away_score"]:

        r_home = "W"
        r_away = "L"

    elif row["home_score"] < row["away_score"]:

        r_home = "L"
        r_away = "W"

    else:

        r_home = "D"
        r_away = "D"

    historial[home].append({

        "gf":
        row["home_score"],

        "gc":
        row["away_score"],

        "resultado":
        r_home

    })

    historial[away].append({

        "gf":
        row["away_score"],

        "gc":
        row["home_score"],

        "resultado":
        r_away

    })

    historial[home] = historial[home][-10:]

    historial[away] = historial[away][-10:]

    # ==========================
    # ACTUALIZAR H2H
    # ==========================

    if row["home_score"] > row["away_score"]:

        winner = home

    elif row["home_score"] < row["away_score"]:

        winner = away

    else:

        winner = "DRAW"

    if clave not in h2h:

        h2h[clave] = []

    h2h[clave].append({

        "winner":
        winner

    })

In [11]:
# ==================================================
# DATAFRAME MODELO
# ==================================================

df_modelo = pd.DataFrame(
    registros
)

df_modelo = df_modelo.fillna(0)

print(df_modelo.shape)

df_modelo.head()

(8000, 38)


,elo_home,elo_away,elo_diff,home_advantage,neutral_match,peso_torneo,gf5_home,gc5_home,wins5_home,draws5_home,...,wins10_away,draws10_away,losses10_away,gd10_away,h2h_home,h2h_away,h2h_draw,home_score,away_score,resultado
0,1500.000000,1500.0,0.000000,0,1,1.0,0.0,0.0,0,0,...,0,0,0,0.0,0,0,0,0.0,0.0,1
1,1500.000000,1500.0,0.000000,0,1,1.0,0.0,0.0,0,0,...,0,0,0,0.0,0,0,0,1.0,0.0,2
2,1500.111166,1500.0,0.111166,0,1,1.0,1.0,0.0,1,0,...,0,1,0,0.0,0,0,0,0.0,0.0,1
3,1500.000000,1500.0,0.000000,0,1,0.6,0.0,0.0,0,0,...,0,0,0,0.0,0,0,0,1.0,1.0,1
4,1500.000000,1500.0,0.000000,0,1,0.6,0.0,0.0,0,0,...,0,1,0,0.0,0,0,0,0.0,1.0,0


In [12]:
# ==================================================
# FEATURES DIFERENCIA
# ==================================================

df_modelo["gf5_diff"] = (
    df_modelo["gf5_home"]
    -
    df_modelo["gf5_away"]
)

df_modelo["gf10_diff"] = (
    df_modelo["gf10_home"]
    -
    df_modelo["gf10_away"]
)

df_modelo["gd5_diff"] = (
    df_modelo["gd5_home"]
    -
    df_modelo["gd5_away"]
)

df_modelo["gd10_diff"] = (
    df_modelo["gd10_home"]
    -
    df_modelo["gd10_away"]
)

df_modelo["wins5_diff"] = (
    df_modelo["wins5_home"]
    -
    df_modelo["wins5_away"]
)

df_modelo["wins10_diff"] = (
    df_modelo["wins10_home"]
    -
    df_modelo["wins10_away"]
)

df_modelo["streak_diff"] = (
    df_modelo["streak_home"]
    -
    df_modelo["streak_away"]
)

print(df_modelo.shape)

(8000, 45)


In [13]:
print(df_modelo.shape)

(8000, 45)


In [14]:
# ==================================================
# SPLIT TEMPORAL
# ==================================================

split_idx = int(
    len(df_modelo) * 0.80
)

train_df = df_modelo.iloc[:split_idx]

test_df = df_modelo.iloc[split_idx:]

print("TRAIN:", train_df.shape)
print("TEST :", test_df.shape)

TRAIN: (6400, 45)
TEST : (1600, 45)


In [15]:
# ==================================================
# FEATURES
# ==================================================

features = [

    "elo_home",
    "elo_away",
    "elo_diff",

    "home_advantage",
    "neutral_match",

    "peso_torneo",

    "gf5_home",
    "gc5_home",
    "wins5_home",
    "draws5_home",
    "losses5_home",
    "gd5_home",

    "gf10_home",
    "gc10_home",
    "wins10_home",
    "draws10_home",
    "losses10_home",
    "gd10_home",

    "gf5_away",
    "gc5_away",
    "wins5_away",
    "draws5_away",
    "losses5_away",
    "gd5_away",

    "gf10_away",
    "gc10_away",
    "wins10_away",
    "draws10_away",
    "losses10_away",
    "gd10_away",

    "streak_home",
    "streak_away",

    "h2h_home",
    "h2h_away",
    "h2h_draw",

    "gf5_diff",
    "gf10_diff",

    "gd5_diff",
    "gd10_diff",

    "wins5_diff",
    "wins10_diff",

    "streak_diff"

]

X_train = train_df[features]
X_test = test_df[features]

In [16]:
# ==================================================
# TARGETS
# ==================================================

y_home_train = train_df["home_score"]
y_home_test = test_df["home_score"]

y_away_train = train_df["away_score"]
y_away_test = test_df["away_score"]

y_result_train = train_df["resultado"]
y_result_test = test_df["resultado"]

In [17]:
# ==================================================
# SCALER
# ==================================================

scaler_poisson = StandardScaler()

X_train_scaled = scaler_poisson.fit_transform(
    X_train
)

X_test_scaled = scaler_poisson.transform(
    X_test
)

In [18]:
# ==================================================
# POISSON HOME
# ==================================================

modelo_home = PoissonRegressor(

    alpha=0.1,

    max_iter=500

)

modelo_home.fit(

    X_train_scaled,

    y_home_train

)

pred_home = modelo_home.predict(

    X_test_scaled

)

mae_home = mean_absolute_error(

    y_home_test,

    pred_home

)

print()

print(
    "MAE HOME:",
    round(mae_home,4)
)


MAE HOME: 1.1985


In [19]:
# ==================================================
# POISSON AWAY
# ==================================================

modelo_away = PoissonRegressor(

    alpha=0.1,

    max_iter=500

)

modelo_away.fit(

    X_train_scaled,

    y_away_train

)

pred_away = modelo_away.predict(

    X_test_scaled

)

mae_away = mean_absolute_error(

    y_away_test,

    pred_away

)

print()

print(
    "MAE AWAY:",
    round(mae_away,4)
)


MAE AWAY: 0.9255


In [ ]:
# ==================================================
# XGBOOST  (con EARLY STOPPING + CALIBRACION)
# ==================================================

from sklearn.calibration import CalibratedClassifierCV

# Validacion temporal INTERNA (ultimo 15% del train) para:
#  - early stopping (evita sobreajuste de los 1000+ arboles)
#  - calibrar las probabilidades
val_cut = int(len(X_train) * 0.85)

X_tr = X_train.iloc[:val_cut]
y_tr = y_result_train.iloc[:val_cut]

X_val = X_train.iloc[val_cut:]
y_val = y_result_train.iloc[val_cut:]

modelo_resultado = XGBClassifier(
    n_estimators=2000,
    max_depth=4,
    learning_rate=0.03,
    subsample=0.8,
    colsample_bytree=0.8,
    min_child_weight=3,
    gamma=0.2,
    objective="multi:softprob",   # softprob -> probabilidades
    num_class=3,
    eval_metric="mlogloss",
    early_stopping_rounds=50,
    random_state=42
)

modelo_resultado.fit(
    X_tr,
    y_tr,
    eval_set=[(X_val, y_val)],
    verbose=False
)

print("Mejor iteracion (early stopping):", modelo_resultado.best_iteration)

# --------------------------------------------------
# CALIBRACION (isotonica) sobre el set de validacion
# --------------------------------------------------
# Las probabilidades crudas de XGBoost suelen estar mal calibradas.
# Usamos FrozenEstimator (sklearn >=1.6) y caemos a cv="prefit" en
# versiones antiguas (asi evitamos el deprecation warning).
try:
    from sklearn.frozen import FrozenEstimator
    modelo_resultado_cal = CalibratedClassifierCV(
        FrozenEstimator(modelo_resultado), method="isotonic"
    )
except Exception:
    modelo_resultado_cal = CalibratedClassifierCV(
        modelo_resultado, method="isotonic", cv="prefit"
    )

modelo_resultado_cal.fit(X_val, y_val)

pred_result = modelo_resultado.predict(X_test)

acc = accuracy_score(
    y_result_test,
    pred_result
)

print()
print("ACCURACY (XGB):", round(acc, 4))


In [21]:
# ==================================================
# IMPORTANCIAS
# ==================================================

importance = pd.DataFrame({

    "feature":features,

    "importance":
    modelo_resultado.feature_importances_

})

importance = (

    importance

    .sort_values(
        "importance",
        ascending=False
    )

)

importance.head(20)

,feature,importance
2,elo_diff,0.052268
38,gd10_diff,0.045550
37,gd5_diff,0.035207
4,neutral_match,0.034594
3,home_advantage,0.033436
32,h2h_home,0.029460
33,h2h_away,0.027570
1,elo_away,0.024106
5,peso_torneo,0.024033
0,elo_home,0.024007


In [ ]:
# ==================================================
# MEJORA 5: EVALUACION COMPARATIVA (cada metodo por separado)
# ==================================================
# Metricas correctas para un modelo de PROBABILIDADES:
#   - LogLoss : penaliza la confianza equivocada     (menor = mejor)
#   - Brier   : error cuadratico de la probabilidad   (menor = mejor)
#   - RPS     : Ranked Probability Score, metrica ESTANDAR del futbol para
#               pronosticos 1X2 (usa el orden visitante<empate<local)  (menor = mejor)
#   - Accuracy: solo de referencia                    (mayor = mejor)

from scipy.stats import poisson
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import log_loss, accuracy_score

CLASSES = [0, 1, 2]  # 0 = gana visitante | 1 = empate | 2 = gana local


def brier_multiclase(y_true, proba, classes=CLASSES):
    y_true = np.asarray(y_true)
    Y = np.zeros((len(y_true), len(classes)))
    for i, c in enumerate(classes):
        Y[:, i] = (y_true == c).astype(float)
    return np.mean(np.sum((proba - Y) ** 2, axis=1))


def rps_multiclase(y_true, proba, classes=CLASSES):
    # Ranked Probability Score (orden ordinal visitante->empate->local)
    y_true = np.asarray(y_true)
    Y = np.zeros((len(y_true), len(classes)))
    for i, c in enumerate(classes):
        Y[:, i] = (y_true == c).astype(float)
    cdf_p = np.cumsum(proba, axis=1)
    cdf_y = np.cumsum(Y, axis=1)
    return np.mean(np.sum((cdf_p - cdf_y) ** 2, axis=1) / (len(classes) - 1))


def evaluar(nombre, y_true, proba):
    proba = np.clip(np.asarray(proba, dtype=float), 1e-15, 1)
    proba = proba / proba.sum(axis=1, keepdims=True)
    pred = np.array(CLASSES)[np.argmax(proba, axis=1)]
    return {
        "Modelo": nombre,
        "LogLoss": log_loss(y_true, proba, labels=CLASSES),
        "Brier": brier_multiclase(y_true, proba),
        "RPS": rps_multiclase(y_true, proba),
        "Accuracy": accuracy_score(y_true, pred),
    }


y_true = y_result_test.values
resultados_eval = []

# --- Baseline 1: clase mayoritaria (referencia minima) ---
p_clase = np.array([np.mean(y_result_train == c) for c in CLASSES])
proba_mayoria = np.tile(p_clase, (len(y_true), 1))
resultados_eval.append(evaluar("Baseline mayoria", y_true, proba_mayoria))

# --- Baseline 2: solo ELO (regresion logistica) ---
feat_base = ["elo_diff", "home_advantage", "neutral_match"]
base_lr = LogisticRegression(max_iter=1000)
base_lr.fit(train_df[feat_base], y_result_train)
proba_base = base_lr.predict_proba(test_df[feat_base])
proba_base = proba_base[:, np.argsort(base_lr.classes_)]  # orden 0,1,2
resultados_eval.append(evaluar("Baseline ELO", y_true, proba_base))

# --- Metodo Poisson + Monte Carlo (independiente) ---
# Convierte las lambdas predichas (pred_home / pred_away) en P(1X2).
def poisson_result_probs(lh, la, max_goals=10):
    h = poisson.pmf(np.arange(max_goals + 1), lh)
    a = poisson.pmf(np.arange(max_goals + 1), la)
    M = np.outer(h, a)
    p_home = np.tril(M, -1).sum()   # local > visitante
    p_draw = np.trace(M)            # empate
    p_away = np.triu(M, 1).sum()    # visitante > local
    s = p_home + p_draw + p_away
    return [p_away / s, p_draw / s, p_home / s]  # orden 0,1,2

proba_poisson = np.array([
    poisson_result_probs(lh, la)
    for lh, la in zip(pred_home, pred_away)
])
resultados_eval.append(evaluar("Poisson (MC)", y_true, proba_poisson))

# --- XGBoost sin calibrar ---
proba_xgb = modelo_resultado.predict_proba(X_test)
resultados_eval.append(evaluar("XGBoost", y_true, proba_xgb))

# --- XGBoost calibrado ---
proba_xgb_cal = modelo_resultado_cal.predict_proba(X_test)
resultados_eval.append(evaluar("XGBoost calibrado", y_true, proba_xgb_cal))

# --- Ensemble (promedio): ELO + Poisson + XGB calibrado ---
# Solo como referencia adicional; arriba se ve cada metodo por separado.
proba_ens = (proba_base + proba_poisson + proba_xgb_cal) / 3
resultados_eval.append(evaluar("Ensemble (promedio)", y_true, proba_ens))

tabla_eval = (
    pd.DataFrame(resultados_eval)
    .sort_values("LogLoss")
    .reset_index(drop=True)
)

print("Comparativa en TEST  (LogLoss/Brier/RPS: menor = mejor | Accuracy: mayor = mejor)\n")
tabla_eval


In [ ]:
# ==================================================
# MEJORA 3 + 4: MODELO DIXON-COLES (ataque/defensa, neutral-aware)
# ==================================================
# (3) Modelamos FUERZA OFENSIVA y DEFENSIVA por seleccion:
#       log(lambda) = c + ataque[equipo] - defensa[rival] + gamma*localia
#     + correccion Dixon-Coles (rho) para marcadores bajos.
# (4) gamma (localia) SOLO se activa si el partido NO es neutral.
# MEJORAS DE METODOLOGIA DE GOLES:
#   - alpha (regularizacion) ELEGIDO por validacion (deviance de Poisson),
#     en vez de fijarlo a ojo -> ratings mas estables para equipos con
#     pocos partidos.
#   - lambda acotado a [0.05, 6] -> evita valores extremos en cruces raros.
#   - misma ventana de recencia que el ELO (TAU_DAYS = 8 años).

from sklearn.linear_model import PoissonRegressor as _PR
from scipy.optimize import minimize_scalar


def _pesos_dc(d):
    fm = d["date"].max()
    pr = np.exp(-(fm - d["date"]).dt.days / 2920)
    t = d["tournament"].astype(str)
    pc = np.where(
        t.eq("Friendly"), 0.45,
        np.where(
            t.str.lower().str.contains("qualification"), 0.85,
            np.where(
                t.isin(["FIFA World Cup", "UEFA Euro", "Copa América",
                        "African Cup of Nations", "AFC Asian Cup", "Gold Cup"]),
                1.15, 0.75)))
    return (pr.values * pc)


def fit_dc(d, alpha=0.05):
    d = d.reset_index(drop=True)
    teams = sorted(set(d["home_team"]) | set(d["away_team"]))

    team_col = pd.concat([d["home_team"], d["away_team"]], ignore_index=True)
    opp_col = pd.concat([d["away_team"], d["home_team"]], ignore_index=True)
    goals = pd.concat([d["home_score"], d["away_score"]], ignore_index=True).astype(int)
    is_home = np.concatenate([
        (1 - d["neutral_match"].values),
        np.zeros(len(d), dtype=int)
    ])

    atk = pd.get_dummies(team_col).reindex(columns=teams, fill_value=0).add_prefix("atk_")
    dfn = pd.get_dummies(opp_col).reindex(columns=teams, fill_value=0).add_prefix("dfn_")
    X = pd.concat([atk, dfn, pd.Series(is_home, name="is_home")], axis=1).astype(float)

    w = _pesos_dc(d)
    m = _PR(alpha=alpha, max_iter=2000)
    m.fit(X, goals.values, sample_weight=np.concatenate([w, w]))

    coef = dict(zip(X.columns, m.coef_))
    c = float(m.intercept_)
    atk_d = {t: coef.get("atk_" + t, 0.0) for t in teams}
    dfn_d = {t: coef.get("dfn_" + t, 0.0) for t in teams}
    gamma = coef.get("is_home", 0.0)

    def lambdas(home, away, neutral):
        hb = 0 if neutral else 1
        lh = np.exp(c + atk_d.get(home, 0.0) + dfn_d.get(away, 0.0) + gamma * hb)
        la = np.exp(c + atk_d.get(away, 0.0) + dfn_d.get(home, 0.0))
        # cota de seguridad: ninguna seleccion espera <0.05 ni >6 goles
        return float(np.clip(lh, 0.05, 6.0)), float(np.clip(la, 0.05, 6.0))

    # --- estimar rho (Dixon-Coles) por MLE sobre los marcadores bajos ---
    atk_h = d["home_team"].map(atk_d).values
    dfn_h = d["home_team"].map(dfn_d).values
    atk_a = d["away_team"].map(atk_d).values
    dfn_a = d["away_team"].map(dfn_d).values
    hb = (1 - d["neutral_match"].values)
    lh = np.exp(c + atk_h + dfn_a + gamma * hb)
    la = np.exp(c + atk_a + dfn_h)
    xs = d["home_score"].values.astype(int)
    ys = d["away_score"].values.astype(int)

    m00 = (xs == 0) & (ys == 0)
    m01 = (xs == 0) & (ys == 1)
    m10 = (xs == 1) & (ys == 0)
    m11 = (xs == 1) & (ys == 1)

    def negll(rho):
        tau = np.ones(len(d))
        tau[m00] = 1 - lh[m00] * la[m00] * rho
        tau[m01] = 1 + lh[m01] * rho
        tau[m10] = 1 + la[m10] * rho
        tau[m11] = 1 - rho
        tau = np.clip(tau, 1e-9, None)
        return -np.sum(w * np.log(tau))

    rho = float(minimize_scalar(negll, bounds=(-0.2, 0.2), method="bounded").x)

    return {"c": c, "atk": atk_d, "dfn": dfn_d, "gamma": gamma,
            "rho": rho, "lambdas": lambdas, "teams": teams}


def dc_score_matrix(pred, home, away, neutral, max_goals=10):
    lh, la = pred["lambdas"](home, away, neutral)
    rho = pred["rho"]
    h = poisson.pmf(np.arange(max_goals + 1), lh)
    a = poisson.pmf(np.arange(max_goals + 1), la)
    M = np.outer(h, a)
    M[0, 0] *= 1 - lh * la * rho
    M[0, 1] *= 1 + lh * rho
    M[1, 0] *= 1 + la * rho
    M[1, 1] *= 1 - rho
    M = np.clip(M, 0, None)
    return M / M.sum()


def dc_result_probs(pred, home, away, neutral, max_goals=10):
    M = dc_score_matrix(pred, home, away, neutral, max_goals)
    p_home = np.tril(M, -1).sum()
    p_draw = np.trace(M)
    p_away = np.triu(M, 1).sum()
    return np.array([p_away, p_draw, p_home])  # orden 0,1,2


# --------------------------------------------------
# Elegir alpha (regularizacion) por validacion temporal
# --------------------------------------------------
def _dc_deviance(pred, d):
    atk, dfn, cc, g = pred["atk"], pred["dfn"], pred["c"], pred["gamma"]
    ah = d["home_team"].map(atk).fillna(0.0).values
    dh = d["home_team"].map(dfn).fillna(0.0).values
    aa = d["away_team"].map(atk).fillna(0.0).values
    da = d["away_team"].map(dfn).fillna(0.0).values
    hb = (1 - d["neutral_match"].values)
    lh = np.clip(np.exp(cc + ah + da + g * hb), 0.05, 6.0)
    la = np.clip(np.exp(cc + aa + dh), 0.05, 6.0)

    def dev(y, lam):
        y = np.asarray(y, float)
        term = np.where(y > 0, y * np.log(y / lam), 0.0)
        return 2.0 * np.sum(term - (y - lam))

    return dev(d["home_score"].values, lh) + dev(d["away_score"].values, la)


def select_alpha_dc(d_tr, d_val, alphas=(0.01, 0.05, 0.15, 0.5)):
    best_a, best_dev = 0.05, np.inf
    for a in alphas:
        try:
            dev = _dc_deviance(fit_dc(d_tr, alpha=a), d_val)
        except Exception:
            continue
        if np.isfinite(dev) and dev < best_dev:
            best_dev, best_a = dev, a
    return best_a


# --------------------------------------------------
# (a) DC para EVALUACION: entrenar en train, medir en test
# --------------------------------------------------
df_train_dc = df.iloc[:split_idx]
df_test_dc = df.iloc[split_idx:]

a_cut = int(split_idx * 0.85)
alpha_dc = select_alpha_dc(df.iloc[:a_cut], df.iloc[a_cut:split_idx])
print("alpha Dixon-Coles elegido por validacion:", alpha_dc)

dc_eval = fit_dc(df_train_dc, alpha=alpha_dc)

proba_dc_eval = np.array([
    dc_result_probs(dc_eval, h, a, bool(n))
    for h, a, n in zip(df_test_dc["home_team"],
                       df_test_dc["away_team"],
                       df_test_dc["neutral_match"])
])

tabla_eval = pd.concat([
    tabla_eval,
    pd.DataFrame([evaluar("Dixon-Coles", y_true, proba_dc_eval)])
], ignore_index=True).sort_values("LogLoss").reset_index(drop=True)

print("rho Dixon-Coles (eval):", round(dc_eval["rho"], 4))
print("\nComparativa ACTUALIZADA (con Dixon-Coles):\n")
print(tabla_eval.to_string())

# --------------------------------------------------
# (b) DC FULL: re-entrenar con TODOS los jugados -> app y simulador
# --------------------------------------------------
dc_full = fit_dc(df, alpha=alpha_dc)
print("\nDixon-Coles FULL entrenado |", len(dc_full["teams"]),
      "equipos | alpha =", alpha_dc,
      "| rho =", round(dc_full["rho"], 4),
      "| gamma =", round(dc_full["gamma"], 4))


In [22]:
# ==================================================
# GUARDAR
# ==================================================
import joblib
joblib.dump(

    modelo_home,

    "modelo_home.joblib"

)

joblib.dump(

    modelo_away,

    "modelo_away.joblib"

)

joblib.dump(

    modelo_resultado,

    "modelo_resultado.joblib"

)

joblib.dump(

    scaler_poisson,

    "scaler_poisson.joblib"

)

print("MODELOS GUARDADOS")

MODELOS GUARDADOS


In [23]:
# ==================================================
# EQUIPOS ACTUALES
# ==================================================

equipos_actuales = []

for equipo in historial:

    ult5 = resumen_equipo(
        historial[equipo][-5:]
    )

    ult10 = resumen_equipo(
        historial[equipo][-10:]
    )

    equipos_actuales.append({

        "PAIS":equipo,

        "ELO":
        elo.get(
            equipo,
            1500
        ),

        "GF5":
        ult5["gf"],

        "GC5":
        ult5["gc"],

        "W5":
        ult5["wins"],

        "D5":
        ult5["draws"],

        "L5":
        ult5["losses"],

        "GF10":
        ult10["gf"],

        "GC10":
        ult10["gc"],

        "W10":
        ult10["wins"],

        "D10":
        ult10["draws"],

        "L10":
        ult10["losses"],

        "STREAK":
        ult5["streak"]

    })

equipos_actuales = pd.DataFrame(
    equipos_actuales
)

equipos_actuales = (

    equipos_actuales

    .sort_values(
        "ELO",
        ascending=False
    )

)

equipos_actuales.to_csv(

    "equipos_actuales.csv",

    index=False

)

print()

print(
    equipos_actuales.head()
)

print()

print(
    "equipos_actuales.csv generado"
)


        PAIS          ELO   GF5  GC5  W5  D5  L5  GF10  GC10  W10  D10  L10  \
91   Morocco  1674.350668  12.0  3.0   3   2   0  19.0   4.0    6    4    0   
110    Spain  1624.676706   9.0  2.0   2   3   0  26.0   4.0    6    4    0   
65   Algeria  1624.370389  11.0  3.0   3   1   1  23.0   5.0    7    2    1   
78   Senegal  1623.996101   8.0  7.0   3   0   2  19.0   9.0    7    1    2   
73   Nigeria  1618.783737   6.0  3.0   2   3   0  20.0   7.0    7    3    0   

     STREAK  
91        0  
110       0  
65        0  
78        0  
73        1  

equipos_actuales.csv generado


In [24]:
print(
    "MAE HOME:",
    round(mae_home,4)
)

MAE HOME: 1.1985


In [25]:
print(
    "MAE AWAY:",
    round(mae_away,4)
)

MAE AWAY: 0.9255


In [26]:
print(
    "ACCURACY:",
    round(acc,4)
)

ACCURACY: 0.5675


In [27]:
equipos_actuales.head(10)

,PAIS,ELO,GF5,GC5,W5,D5,L5,GF10,GC10,W10,D10,L10,STREAK
91,Morocco,1674.350668,12.0,3.0,3,2,0,19.0,4.0,6,4,0,0
110,Spain,1624.676706,9.0,2.0,2,3,0,26.0,4.0,6,4,0,0
65,Algeria,1624.370389,11.0,3.0,3,1,1,23.0,5.0,7,2,1,0
78,Senegal,1623.996101,8.0,7.0,3,0,2,19.0,9.0,7,1,2,0
73,Nigeria,1618.783737,6.0,3.0,2,3,0,20.0,7.0,7,3,0,1
105,Japan,1604.053962,8.0,2.0,4,1,0,15.0,8.0,6,3,1,0
95,Argentina,1603.093811,16.0,1.0,5,0,0,25.0,4.0,8,1,1,5
89,England,1602.368143,10.0,2.0,3,1,1,22.0,5.0,7,1,2,0
119,Ivory Coast,1601.712073,11.0,3.0,4,0,1,18.0,7.0,7,1,2,3
19,Mexico,1598.847280,6.0,1.0,3,2,0,13.0,3.0,6,3,1,3


In [28]:
equipos_actuales.columns

Index(['PAIS', 'ELO', 'GF5', 'GC5', 'W5', 'D5', 'L5', 'GF10', 'GC10', 'W10',
       'D10', 'L10', 'STREAK'],
      dtype='object')

In [29]:
# ==================================================
# MÓDULO 2: GENERADOR AUTOMÁTICO DE PARTIDO
# ==================================================

!pip install -q ipywidgets openpyxl

import ipywidgets as widgets
from IPython.display import display
from google.colab import files

# ==================================================
# CARGAR EQUIPOS ACTUALES
# ==================================================

equipos_actuales = pd.read_csv("equipos_actuales.csv")

equipos_disponibles = (
    equipos_actuales["PAIS"]
    .sort_values()
    .tolist()
)

# ==================================================
# DROPDOWNS
# ==================================================

dropdown_local = widgets.Dropdown(
    options=equipos_disponibles,
    description="Local:",
    layout=widgets.Layout(width="400px")
)

dropdown_visitante = widgets.Dropdown(
    options=equipos_disponibles,
    description="Visitante:",
    layout=widgets.Layout(width="400px")
)

boton_generar = widgets.Button(
    description="Generar partido.xlsx",
    button_style="success"
)

salida = widgets.Output()

display(dropdown_local)
display(dropdown_visitante)
display(boton_generar)
display(salida)

# ==================================================
# FUNCIÓN PARA GENERAR EXCEL
# ==================================================

def generar_partido_excel(local, visitante):

    if local == visitante:
        raise ValueError("El equipo local y visitante no pueden ser el mismo.")

    datos_local = equipos_actuales[
        equipos_actuales["PAIS"] == local
    ].iloc[0]

    datos_visitante = equipos_actuales[
        equipos_actuales["PAIS"] == visitante
    ].iloc[0]

    hoja_local = pd.DataFrame([{

        "PAIS": local,

        "ELO": datos_local["ELO"],

        "GOLES_FAVOR_5": datos_local["GF5"],
        "GOLES_CONTRA_5": datos_local["GC5"],

        "GANADOS_5": datos_local["W5"],
        "EMPATADOS_5": datos_local["D5"],
        "PERDIDOS_5": datos_local["L5"],

        "GOLES_FAVOR_10": datos_local["GF10"],
        "GOLES_CONTRA_10": datos_local["GC10"],

        "GANADOS_10": datos_local["W10"],
        "EMPATADOS_10": datos_local["D10"],
        "PERDIDOS_10": datos_local["L10"],

        "STREAK": datos_local["STREAK"],

        "LOCAL": 1

    }])

    hoja_visitante = pd.DataFrame([{

        "PAIS": visitante,

        "ELO": datos_visitante["ELO"],

        "GOLES_FAVOR_5": datos_visitante["GF5"],
        "GOLES_CONTRA_5": datos_visitante["GC5"],

        "GANADOS_5": datos_visitante["W5"],
        "EMPATADOS_5": datos_visitante["D5"],
        "PERDIDOS_5": datos_visitante["L5"],

        "GOLES_FAVOR_10": datos_visitante["GF10"],
        "GOLES_CONTRA_10": datos_visitante["GC10"],

        "GANADOS_10": datos_visitante["W10"],
        "EMPATADOS_10": datos_visitante["D10"],
        "PERDIDOS_10": datos_visitante["L10"],

        "STREAK": datos_visitante["STREAK"],

        "LOCAL": 0

    }])

    nombre_archivo = "partido.xlsx"

    with pd.ExcelWriter(nombre_archivo, engine="openpyxl") as writer:

        hoja_local.to_excel(
            writer,
            sheet_name=local[:31],
            index=False
        )

        hoja_visitante.to_excel(
            writer,
            sheet_name=visitante[:31],
            index=False
        )

    return nombre_archivo

# ==================================================
# ACCIÓN DEL BOTÓN
# ==================================================

def on_click_generar(b):

    with salida:

        salida.clear_output()

        local = dropdown_local.value
        visitante = dropdown_visitante.value

        try:

            archivo = generar_partido_excel(
                local,
                visitante
            )

            print("Archivo generado correctamente:")
            print(archivo)
            print()
            print("Local:", local)
            print("Visitante:", visitante)

            files.download(archivo)

        except Exception as e:

            print("Error:", e)

boton_generar.on_click(on_click_generar)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 15.6 MB/s eta 0:00:00


Dropdown(description='Local:', layout=Layout(width='400px'), options=('Abkhazia', 'Afghanistan', 'Albania', 'A…

Dropdown(description='Visitante:', layout=Layout(width='400px'), options=('Abkhazia', 'Afghanistan', 'Albania'…

Button(button_style='success', description='Generar partido.xlsx', style=ButtonStyle())

Output()

In [30]:
# ==================================================
# CONSTRUIR FEATURES PARA UN PARTIDO
# ==================================================

def construir_features_partido(local, visitante, neutral_match, peso_torneo):

    datos_local = equipos_actuales[
        equipos_actuales["PAIS"] == local
    ].iloc[0]

    datos_visitante = equipos_actuales[
        equipos_actuales["PAIS"] == visitante
    ].iloc[0]

    home_advantage = 1 - neutral_match

    elo_home = datos_local["ELO"]
    elo_away = datos_visitante["ELO"]
    elo_diff = elo_home - elo_away

    gf5_home = datos_local["GF5"]
    gc5_home = datos_local["GC5"]
    wins5_home = datos_local["W5"]
    draws5_home = datos_local["D5"]
    losses5_home = datos_local["L5"]
    gd5_home = gf5_home - gc5_home

    gf10_home = datos_local["GF10"]
    gc10_home = datos_local["GC10"]
    wins10_home = datos_local["W10"]
    draws10_home = datos_local["D10"]
    losses10_home = datos_local["L10"]
    gd10_home = gf10_home - gc10_home

    gf5_away = datos_visitante["GF5"]
    gc5_away = datos_visitante["GC5"]
    wins5_away = datos_visitante["W5"]
    draws5_away = datos_visitante["D5"]
    losses5_away = datos_visitante["L5"]
    gd5_away = gf5_away - gc5_away

    gf10_away = datos_visitante["GF10"]
    gc10_away = datos_visitante["GC10"]
    wins10_away = datos_visitante["W10"]
    draws10_away = datos_visitante["D10"]
    losses10_away = datos_visitante["L10"]
    gd10_away = gf10_away - gc10_away

    streak_home = datos_local["STREAK"]
    streak_away = datos_visitante["STREAK"]

    # H2H actual usando el diccionario creado en entrenamiento
    clave = tuple(sorted([local, visitante]))
    historial_h2h = h2h.get(clave, [])

    h2h_home, h2h_away, h2h_draw = obtener_h2h(
        historial_h2h,
        local,
        visitante
    )

    X_partido = pd.DataFrame([{

        "elo_home": elo_home,
        "elo_away": elo_away,
        "elo_diff": elo_diff,

        "home_advantage": home_advantage,
        "neutral_match": neutral_match,

        "peso_torneo": peso_torneo,

        "gf5_home": gf5_home,
        "gc5_home": gc5_home,
        "wins5_home": wins5_home,
        "draws5_home": draws5_home,
        "losses5_home": losses5_home,
        "gd5_home": gd5_home,

        "gf10_home": gf10_home,
        "gc10_home": gc10_home,
        "wins10_home": wins10_home,
        "draws10_home": draws10_home,
        "losses10_home": losses10_home,
        "gd10_home": gd10_home,

        "gf5_away": gf5_away,
        "gc5_away": gc5_away,
        "wins5_away": wins5_away,
        "draws5_away": draws5_away,
        "losses5_away": losses5_away,
        "gd5_away": gd5_away,

        "gf10_away": gf10_away,
        "gc10_away": gc10_away,
        "wins10_away": wins10_away,
        "draws10_away": draws10_away,
        "losses10_away": losses10_away,
        "gd10_away": gd10_away,

        "streak_home": streak_home,
        "streak_away": streak_away,

        "h2h_home": h2h_home,
        "h2h_away": h2h_away,
        "h2h_draw": h2h_draw,

        "gf5_diff": gf5_home - gf5_away,
        "gf10_diff": gf10_home - gf10_away,

        "gd5_diff": gd5_home - gd5_away,
        "gd10_diff": gd10_home - gd10_away,

        "wins5_diff": wins5_home - wins5_away,
        "wins10_diff": wins10_home - wins10_away,

        "streak_diff": streak_home - streak_away

    }])

    return X_partido[features]

In [31]:
# ==================================================
# MONTE CARLO
# ==================================================

def montecarlo_partido(lambda_local, lambda_visitante, N=100000):

    goles_local = np.random.poisson(
        lambda_local,
        N
    )

    goles_visitante = np.random.poisson(
        lambda_visitante,
        N
    )

    p_local = np.mean(
        goles_local > goles_visitante
    )

    p_empate = np.mean(
        goles_local == goles_visitante
    )

    p_visitante = np.mean(
        goles_local < goles_visitante
    )

    resultados = pd.DataFrame({

        "GOLES_LOCAL": goles_local,
        "GOLES_VISITANTE": goles_visitante

    })

    marcadores = (

        resultados
        .groupby([
            "GOLES_LOCAL",
            "GOLES_VISITANTE"
        ])
        .size()
        .reset_index(name="FRECUENCIA")
        .sort_values(
            "FRECUENCIA",
            ascending=False
        )

    )

    marcadores["PROB"] = (
        marcadores["FRECUENCIA"] / N
    )

    return (
        p_local,
        p_empate,
        p_visitante,
        marcadores.head(10)
    )

In [ ]:
# ==================================================
# MÓDULO 3: SIMULAR PARTIDO
# ==================================================

import numpy as np
import pandas as pd
import ipywidgets as widgets
from IPython.display import display

# ==================================================
# CARGAR EQUIPOS
# ==================================================

equipos_actuales = pd.read_csv("equipos_actuales.csv")

equipos_disponibles = (
    equipos_actuales["PAIS"]
    .sort_values()
    .tolist()
)

# ==================================================
# WIDGETS
# ==================================================

local_widget = widgets.Dropdown(
    options=equipos_disponibles,
    description="Local:",
    layout=widgets.Layout(width="400px")
)

visitante_widget = widgets.Dropdown(
    options=equipos_disponibles,
    description="Visitante:",
    layout=widgets.Layout(width="400px")
)

neutral_widget = widgets.Dropdown(
    options=[("No", 0), ("Sí", 1)],
    description="Neutral:",
    layout=widgets.Layout(width="400px")
)

torneo_widget = widgets.Dropdown(
    options=[
        ("Mundial", 2.0),
        ("Euro / Copa América", 1.8),
        ("Eliminatoria Mundialista", 1.5),
        ("Nations League", 1.3),
        ("Copa Regional", 1.3),
        ("Amistoso", 0.7),
        ("Otro", 1.0)
    ],
    description="Torneo:",
    layout=widgets.Layout(width="400px")
)

boton_simular = widgets.Button(
    description="Simular partido",
    button_style="success"
)

salida = widgets.Output()

display(local_widget)
display(visitante_widget)
display(neutral_widget)
display(torneo_widget)
display(boton_simular)
display(salida)


# ==================================================
# SIMULAR CON BOTÓN
# ==================================================
# antes de dar simular partido correr este codigo
def simular_click(b):

    with salida:

        salida.clear_output()

        local = local_widget.value
        visitante = visitante_widget.value
        neutral_match = neutral_widget.value
        peso_torneo = torneo_widget.value

        if local == visitante:
            print("Error: el local y visitante no pueden ser el mismo equipo.")
            return

        X_partido = construir_features_partido(
            local,
            visitante,
            neutral_match,
            peso_torneo
        )

        X_scaled = scaler_poisson.transform(
            X_partido
        )

        lambda_local = modelo_home.predict(
            X_scaled
        )[0]

        lambda_visitante = modelo_away.predict(
            X_scaled
        )[0]

        p_local, p_empate, p_visitante, marcadores = montecarlo_partido(
            lambda_local,
            lambda_visitante,
            N=100000
        )

        # XGBoost SIN calibrar y CALIBRADO (mostramos ambos)
        proba_xgb = modelo_resultado.predict_proba(X_partido)[0]
        proba_xgb_cal = modelo_resultado_cal.predict_proba(X_partido)[0]

        # Dixon-Coles (ataque/defensa, neutral-aware)
        proba_dc = dc_result_probs(
            dc_full, local, visitante, bool(neutral_match)
        )
        lh_dc, la_dc = dc_full["lambdas"](local, visitante, bool(neutral_match))

        print("=" * 50)
        print(f"{local} vs {visitante}")
        print("=" * 50)

        print()
        print("GOLES ESPERADOS")
        print("  Poisson(42 feats):", local, round(lambda_local, 3),
              "|", visitante, round(lambda_visitante, 3))
        print("  Dixon-Coles      :", local, round(lh_dc, 3),
              "|", visitante, round(la_dc, 3))

        print()
        print("--- MÉTODO 1: Poisson + Monte Carlo ---")
        print("Gana", local, ":", round(p_local * 100, 2), "%")
        print("Empate:", round(p_empate * 100, 2), "%")
        print("Gana", visitante, ":", round(p_visitante * 100, 2), "%")

        print()
        print("--- MÉTODO 2: XGBoost (SIN calibrar) ---")
        print("Gana", local, ":", round(proba_xgb[2] * 100, 2), "%")
        print("Empate:", round(proba_xgb[1] * 100, 2), "%")
        print("Gana", visitante, ":", round(proba_xgb[0] * 100, 2), "%")

        print()
        print("--- MÉTODO 3: XGBoost (CALIBRADO) ---")
        print("Gana", local, ":", round(proba_xgb_cal[2] * 100, 2), "%")
        print("Empate:", round(proba_xgb_cal[1] * 100, 2), "%")
        print("Gana", visitante, ":", round(proba_xgb_cal[0] * 100, 2), "%")

        print()
        print("--- MÉTODO 4: Dixon-Coles ---")
        print("Gana", local, ":", round(proba_dc[2] * 100, 2), "%")
        print("Empate:", round(proba_dc[1] * 100, 2), "%")
        print("Gana", visitante, ":", round(proba_dc[0] * 100, 2), "%")

        # ENSEMBLE (promedio de DC + XGB calibrado + MC)
        proba_mc = np.array([p_visitante, p_empate, p_local])
        proba_ens = (proba_dc + proba_xgb_cal + proba_mc) / 3
        print()
        print("--- ENSEMBLE (promedio de 3) ---")
        print("Gana", local, ":", round(proba_ens[2] * 100, 2), "%")
        print("Empate:", round(proba_ens[1] * 100, 2), "%")
        print("Gana", visitante, ":", round(proba_ens[0] * 100, 2), "%")

        print()
        print("MARCADORES MÁS PROBABLES (Poisson MC)")
        display(marcadores)

boton_simular.on_click(simular_click)


In [ ]:
# ==================================================
# MEJORA 6: SIMULADOR DEL MUNDIAL 2026 (bracket OFICIAL)
# ==================================================
# Simula el torneo COMPLETO miles de veces con Dixon-Coles como motor:
#   - Reconstruye los 12 grupos del calendario del CSV.
#   - Respeta los partidos ya jugados; simula los que faltan.
#   - Clasifican 1o y 2o de cada grupo + 8 mejores terceros.
#   - Usa el CUADRO OFICIAL FIFA 2026 (partidos 73-104), incluida la
#     tabla de elegibilidad de terceros (asignacion por emparejamiento).
#   - Empate en eliminatoria -> "penales" segun fuerza relativa.

import collections

N_SIM = 10000
np.random.seed(RANDOM_STATE)
rng = np.random.default_rng(RANDOM_STATE)

# --------------------------------------------------
# Calendario del Mundial (jugados + por jugar) y grupos
# --------------------------------------------------
wc_jugados = df[(df["tournament"] == "FIFA World Cup")
                & (df["date"] >= "2026-01-01")]
wc_fixtures = pd.concat([wc_jugados, por_jugar], ignore_index=True)

adj = collections.defaultdict(set)
for _, r in wc_fixtures.iterrows():
    adj[r["home_team"]].add(r["away_team"])
    adj[r["away_team"]].add(r["home_team"])

teams_wc = sorted(pd.unique(pd.concat([wc_fixtures["home_team"],
                                       wc_fixtures["away_team"]])))
seen, groups, gid = set(), {}, 0
for t in teams_wc:
    if t in seen:
        continue
    stack, comp = [t], set()
    while stack:
        x = stack.pop()
        if x in seen:
            continue
        seen.add(x); comp.add(x); stack.extend(adj[x] - seen)
    groups[chr(65 + gid)] = sorted(comp); gid += 1

team2group = {t: g for g, ts in groups.items() for t in ts}

print("Grupos reconstruidos:", len(groups))
for g, ts in groups.items():
    print(" ", g, ts)

fixtures = []
for _, r in wc_fixtures.iterrows():
    gh = None if pd.isna(r["home_score"]) else int(r["home_score"])
    ga = None if pd.isna(r["away_score"]) else int(r["away_score"])
    fixtures.append((r["home_team"], r["away_team"], bool(r["neutral"]), gh, ga))

# --------------------------------------------------
# CUADRO OFICIAL FIFA 2026 (partidos 73-104)
# --------------------------------------------------
R32 = {
    73: ("RU_A", "RU_B"), 74: ("W_E", "3RD"), 75: ("W_F", "RU_C"),
    76: ("W_C", "RU_F"), 77: ("W_I", "3RD"), 78: ("RU_E", "RU_I"),
    79: ("W_A", "3RD"), 80: ("W_L", "3RD"), 81: ("W_D", "3RD"),
    82: ("W_G", "3RD"), 83: ("RU_K", "RU_L"), 84: ("W_H", "RU_J"),
    85: ("W_B", "3RD"), 86: ("W_J", "RU_H"), 87: ("W_K", "3RD"),
    88: ("RU_D", "RU_G"),
}
THIRD_ELIG = {
    74: set("ABCDF"), 77: set("CDFGH"), 79: set("CEFHI"), 80: set("EHIJK"),
    81: set("BEFIJ"), 82: set("AEHIJ"), 85: set("EFGIJ"), 87: set("DEIJL"),
}
R16 = {89: (74, 77), 90: (73, 75), 91: (76, 78), 92: (79, 80),
       93: (83, 84), 94: (81, 82), 95: (86, 88), 96: (85, 87)}
QF = {97: (89, 90), 98: (93, 94), 99: (91, 92), 100: (95, 96)}
SF = {101: (97, 98), 102: (99, 100)}
FINAL = (101, 102)

STAGE = {"GROUP": 0, "R32": 1, "R16": 2, "QF": 3, "SF": 4, "FINAL": 5, "CHAMP": 6}


def match_thirds(qual_groups):
    """Asigna los 8 grupos con tercero clasificado a los slots '3RD'
    respetando la elegibilidad oficial (emparejamiento bipartito)."""
    R = {}

    def assign(slot, vis):
        for g in THIRD_ELIG[slot]:
            if g in qual_groups and g not in vis:
                vis.add(g)
                if g not in R or assign(R[g], vis):
                    R[g] = slot
                    return True
        return False

    for s in THIRD_ELIG:
        assign(s, set())
    slot2group = {s: g for g, s in R.items()}
    libres_s = [s for s in THIRD_ELIG if s not in slot2group]
    libres_g = [g for g in qual_groups if g not in R]
    for s, g in zip(libres_s, libres_g):
        slot2group[s] = g
    return slot2group


def sim_score(h, a, neu):
    lh, la = dc_full["lambdas"](h, a, neu)
    return rng.poisson(lh), rng.poisson(la)


def ko_winner(h, a):
    gh, ga = sim_score(h, a, True)
    if gh > ga:
        return h
    if ga > gh:
        return a
    lh, la = dc_full["lambdas"](h, a, True)
    return h if rng.random() < lh / (lh + la) else a


def standings(gt, results):
    st = {t: [0, 0, 0] for t in gt}  # pts, gf, gc
    for h, a, gh, ga in results:
        st[h][1] += gh; st[h][2] += ga; st[a][1] += ga; st[a][2] += gh
        if gh > ga:
            st[h][0] += 3
        elif ga > gh:
            st[a][0] += 3
        else:
            st[h][0] += 1; st[a][0] += 1
    rows = [(t, st[t][0], st[t][1] - st[t][2], st[t][1], rng.random()) for t in gt]
    rows.sort(key=lambda x: (-x[1], -x[2], -x[3], x[4]))
    return rows


# --------------------------------------------------
# Monte Carlo del torneo
# --------------------------------------------------
cont = {t: collections.Counter() for t in teams_wc}
for _ in range(N_SIM):
    gres = {g: [] for g in groups}
    for h, a, neu, gh, ga in fixtures:
        if gh is None:
            gh, ga = sim_score(h, a, neu)
        gres[team2group[h]].append((h, a, gh, ga))

    W, RU, thirds = {}, {}, []
    for g in groups:
        o = standings(groups[g], gres[g])
        W[g], RU[g] = o[0][0], o[1][0]
        thirds.append((g, o[2][0], o[2][1], o[2][2], o[2][3], o[2][4]))
    thirds.sort(key=lambda x: (-x[2], -x[3], -x[4], x[5]))
    third_team = {x[0]: x[1] for x in thirds[:8]}
    slot2group = match_thirds(set(third_team))

    def team_of(tok, mnum):
        if tok == "3RD":
            return third_team[slot2group[mnum]]
        kind, g = tok.split("_")
        return W[g] if kind == "W" else RU[g]

    reached = {t: 0 for t in teams_wc}
    win = {}
    for mnum, (t1, t2) in R32.items():
        a1, a2 = team_of(t1, mnum), team_of(t2, mnum)
        reached[a1] = max(reached[a1], STAGE["R32"])
        reached[a2] = max(reached[a2], STAGE["R32"])
        w = ko_winner(a1, a2); win[mnum] = w
        reached[w] = STAGE["R16"]
    for mnum, (m1, m2) in R16.items():
        w = ko_winner(win[m1], win[m2]); win[mnum] = w
        reached[w] = STAGE["QF"]
    for mnum, (m1, m2) in QF.items():
        w = ko_winner(win[m1], win[m2]); win[mnum] = w
        reached[w] = STAGE["SF"]
    for mnum, (m1, m2) in SF.items():
        w = ko_winner(win[m1], win[m2]); win[mnum] = w
        reached[w] = STAGE["FINAL"]
    champ = ko_winner(win[FINAL[0]], win[FINAL[1]])
    reached[champ] = STAGE["CHAMP"]

    for t in teams_wc:
        st = reached[t]
        for nombre, rank in STAGE.items():
            if rank >= 1 and st >= rank:
                cont[t][nombre] += 1

filas = []
for t in teams_wc:
    c = cont[t]
    filas.append({
        "PAIS": t, "GRUPO": team2group[t], "ELO": round(elo.get(t, 1500)),
        "P_Clasifica": 100 * c["R32"] / N_SIM,   # pasa de grupos
        "P_Octavos": 100 * c["R16"] / N_SIM,     # llega a octavos (16)
        "P_Cuartos": 100 * c["QF"] / N_SIM,      # llega a cuartos (8)
        "P_Semis": 100 * c["SF"] / N_SIM,        # llega a semis (4)
        "P_Final": 100 * c["FINAL"] / N_SIM,     # juega la final (2)
        "P_CAMPEON": 100 * c["CHAMP"] / N_SIM,
    })

prob_mundial = (pd.DataFrame(filas)
                .sort_values("P_CAMPEON", ascending=False)
                .reset_index(drop=True)
                .round(2))

prob_mundial.to_csv("probabilidades_mundial.csv", index=False)

print(f"\n=== PROBABILIDADES MUNDIAL 2026 (bracket oficial, {N_SIM:,} sims) ===\n")
print(prob_mundial.head(20).to_string(index=False))
print("\nprobabilidades_mundial.csv generado")
